# Week 6 Exercise — Emmanuel

Fine-tuning exercise using the **Student Placement Prediction** dataset from Kaggle. Download the dataset, prepare it for fine-tuning, and train a model.

**Requirements:** `OPENAI_API_KEY` for fine-tuning; Kaggle credentials for dataset download (`KAGGLE_USERNAME`, `KAGGLE_KEY` or `~/.kaggle/kaggle.json`).

In [ ]:
# Imports for fine-tuning
import os
import re
import json
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
openai_client = OpenAI() if os.getenv("OPENAI_API_KEY") else None

In [ ]:
# Install kagglehub if not already installed
%pip install -q kagglehub

In [ ]:
import kagglehub

# Download latest version of Student Placement Prediction dataset
path = kagglehub.dataset_download("suhanigupta04/student-placement-prediction-dataset")

print("Path to dataset files:", path)

In [ ]:
# List downloaded files
list(Path(path).rglob("*")) if path else []

In [ ]:
def load_dataset_from_path(data_path):
    """Load all CSV files from the downloaded dataset path into a single DataFrame."""
    data_path = Path(data_path)
    csv_files = list(data_path.rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in {data_path}")
    dfs = []
    for f in csv_files:
        df = pd.read_csv(f)
        df["_source"] = f.name
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True) if len(dfs) > 1 else dfs[0].drop(columns=["_source"], errors="ignore")


def clean_dataset(df):
    """
    Clean the student placement dataset:
    - Strip whitespace from string columns
    - Drop duplicate rows
    - Drop rows with all-NaN
    - Fill or drop missing values (numeric: median; categorical: 'Unknown')
    - Normalize column names (strip, lowercase, replace spaces with underscores)
    """
    df = df.copy()
    df = df.drop(columns=["_source"], errors="ignore")
    # Normalize column names
    df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]
    # Strip whitespace from string columns
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].astype(str).str.strip().replace("nan", pd.NA)
    # Drop fully empty rows
    df = df.dropna(how="all")
    # Drop duplicates
    df = df.drop_duplicates()
    # Handle missing values: numeric -> median, object -> "Unknown"
    for col in df.columns:
        if df[col].dtype in ["float64", "int64"]:
            if df[col].isna().any():
                df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna("Unknown")
    return df


# Load and clean
df_raw = load_dataset_from_path(path)
print("Raw shape:", df_raw.shape)
print("Raw columns:", list(df_raw.columns))

df_clean = clean_dataset(df_raw)
print("\nCleaned shape:", df_clean.shape)
df_clean.head()

In [ ]:
# Save cleaned dataset for downstream use
OUTPUT_PATH = Path("data/student_placement_cleaned.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(OUTPUT_PATH, index=False)
print(f"Saved cleaned dataset to {OUTPUT_PATH}")

In [ ]:
# Features and target
FEATURE_COLS = [
    "branch",
    "college_tier",
    "cgpa",
    "coding_skills",
    "aptitude_score",
    "communication_skills",
    "ml_knowledge",
]
TARGET_COL = "salary_package_lpa"

# Select only features and target
df_model = df_clean[FEATURE_COLS + [TARGET_COL]].copy()
df_model = df_model.dropna(subset=FEATURE_COLS + [TARGET_COL])

# Split into train (80%), validation (10%), test (10%)
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

train_df, temp_df = train_test_split(df_model, test_size=0.4, random_state=RANDOM_STATE)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=RANDOM_STATE)

print(f"Train: {len(train_df):,} samples ({len(train_df)/len(df_model)*100:.1f}%)")
print(f"Validation: {len(val_df):,} samples ({len(val_df)/len(df_model)*100:.1f}%)")
print(f"Test: {len(test_df):,} samples ({len(test_df)/len(df_model)*100:.1f}%)")
print(f"\nFeatures: {FEATURE_COLS}")
print(f"Target: {TARGET_COL}")

In [ ]:
# Separate features (X) and target (y) for each split
X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET_COL]
X_val = val_df[FEATURE_COLS]
y_val = val_df[TARGET_COL]
X_test = test_df[FEATURE_COLS]
y_test = test_df[TARGET_COL]

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

In [ ]:
# Prepare data for OpenAI fine-tuning (JSONL format)
# Reference: Week 6 Day 5 — each line: {"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}


def messages_for(row):
    """Format a student row into user/assistant messages for fine-tuning."""
    prompt = (
        "Predict the salary package (LPA) for this student. Respond with the LPA value only, no explanation.\n\n"
        f"Branch: {row['branch']}\n"
        f"College Tier: {row['college_tier']}\n"
        f"CGPA: {row['cgpa']}\n"
        f"Coding Skills: {row['coding_skills']}\n"
        f"Aptitude Score: {row['aptitude_score']}\n"
        f"Communication Skills: {row['communication_skills']}\n"
        f"ML Knowledge: {row['ml_knowledge']}"
    )
    salary = row[TARGET_COL]
    return [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": f"{salary:.2f} LPA"},
    ]


def make_jsonl(df):
    """Convert DataFrame rows to JSONL string for OpenAI fine-tuning."""
    lines = []
    for _, row in df.iterrows():
        messages = messages_for(row)
        lines.append(json.dumps({"messages": messages}))
    return "\n".join(lines)


def write_jsonl(df, filepath):
    """Write DataFrame to JSONL file."""
    Path(filepath).parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, "w") as f:
        f.write(make_jsonl(df))
    print(f"Wrote {len(df):,} examples to {filepath}")

In [ ]:
# OpenAI recommends 50–100 examples for fine-tuning; use a subset to keep cost low
FINE_TUNE_TRAIN_SIZE = 100
FINE_TUNE_VAL_SIZE = 50

fine_tune_train = train_df.sample(n=min(FINE_TUNE_TRAIN_SIZE, len(train_df)), random_state=RANDOM_STATE)
fine_tune_val = val_df.sample(n=min(FINE_TUNE_VAL_SIZE, len(val_df)), random_state=RANDOM_STATE)

JSONL_DIR = Path("jsonl")
write_jsonl(fine_tune_train, JSONL_DIR / "fine_tune_train.jsonl")
write_jsonl(fine_tune_val, JSONL_DIR / "fine_tune_validation.jsonl")

In [ ]:
# Upload JSONL files to OpenAI for fine-tuning (requires OPENAI_API_KEY)
if openai_client:
    with open(JSONL_DIR / "fine_tune_train.jsonl", "rb") as f:
        train_file = openai_client.files.create(file=f, purpose="fine-tune")
    with open(JSONL_DIR / "fine_tune_validation.jsonl", "rb") as f:
        validation_file = openai_client.files.create(file=f, purpose="fine-tune")
    print(f"Train file ID: {train_file.id}")
    print(f"Validation file ID: {validation_file.id}")
else:
    print("OPENAI_API_KEY not set. Skipping upload. Set the key and re-run to upload.")

In [ ]:
# Create fine-tuning job (run after upload cell succeeds)
# Uses gpt-4.1-nano as in Day 5; adjust model/suffix as needed
if openai_client:
    job = openai_client.fine_tuning.jobs.create(
        training_file=train_file.id,
        validation_file=validation_file.id,
        model="gpt-4.1-nano-2025-04-14",
        seed=RANDOM_STATE,
        hyperparameters={"n_epochs": 1, "batch_size": 1},
        suffix="placement",
    )
    print(f"Fine-tuning job created: {job.id}")
    print("Monitor at: https://platform.openai.com/finetune")
else:
    print("OPENAI_API_KEY not set. Run the upload cell first.")

In [ ]:
job.id

In [ ]:
ft_job = openai_client.fine_tuning.jobs.retrieve(job.id)
fine_tuned_model_name = ft_job.fine_tuned_model
print(f"Fine-tuned model: {fine_tuned_model_name}")

In [ ]:
# Evaluation: custom evaluator for placement (LPA)
from evaluator import evaluate

EVAL_SIZE = 200


def prompt_for(row):
    """Build user prompt for inference (same format as training)."""
    return (
        "Predict the salary package (LPA) for this student. Respond with the LPA value only, no explanation.\n\n"
        f"Branch: {row['branch']}\n"
        f"College Tier: {row['college_tier']}\n"
        f"CGPA: {row['cgpa']}\n"
        f"Coding Skills: {row['coding_skills']}\n"
        f"Aptitude Score: {row['aptitude_score']}\n"
        f"Communication Skills: {row['communication_skills']}\n"
        f"ML Knowledge: {row['ml_knowledge']}"
    )


def placement_predictor(row):
    """Call fine-tuned model to predict salary for a student row."""
    response = openai_client.chat.completions.create(
        model=fine_tuned_model_name,
        messages=[{"role": "user", "content": prompt_for(row)}],
        max_tokens=10,
    )
    return response.choices[0].message.content or "0"

In [ ]:
# Run evaluation
evaluate(placement_predictor, test_df, target_col=TARGET_COL, size=EVAL_SIZE, random_state=RANDOM_STATE)